In [1]:
import gradio as gr
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import socket
import subprocess
import threading
import time
import sys
import io
from IPython.display import display, HTML

print = lambda *args, **kwargs: __builtins__.print(*args, **kwargs, flush=True)

def kill_port(port):
    """Освобождает порт"""
    try:
        cmd = f'netstat -ano | findstr :{port}'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.stdout:
            for line in result.stdout.strip().split('\n'):
                if f':{port}' in line and 'LISTENING' in line:
                    pid = line.strip().split()[-1]
                    subprocess.run(f'taskkill /F /PID {pid}', shell=True, capture_output=True)
                    print(f"Порт {port} освобожден")
    except Exception as e:
        print(f"Ошибка при освобождении порта {port}: {e}")

def get_free_port(start=7860):
    """Находит свободный порт"""
    for port in range(start, start + 5):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                if s.connect_ex(('127.0.0.1', port)) != 0:
                    return port
                kill_port(port)
        except:
            continue
    return start

print("Импорты и функции загружены")

C:\Users\Битрикс\.conda\envs\JupyterProject\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Импорты и функции загружены


In [2]:
def get_stock_data(ticker, period="5y"):
    """Получает исторические данные по акции"""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period=period)

        if hist.empty:
            return None, f"Нет данных для тикера {ticker}"

        hist['MA20'] = hist['Close'].rolling(window=20).mean()
        hist['MA50'] = hist['Close'].rolling(window=50).mean()
        hist['MA200'] = hist['Close'].rolling(window=200).mean()

        return hist, None
    except Exception as e:
        return None, f"Ошибка получения данных: {str(e)}"

def create_stock_chart(hist, ticker, chart_type="candlestick"):
    """Создает график акции"""
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.6, 0.2, 0.2],
        subplot_titles=(f'{ticker} - Цена', 'Объем', 'RSI')
    )

    if chart_type == "candlestick":
        fig.add_trace(
            go.Candlestick(
                x=hist.index,
                open=hist['Open'],
                high=hist['High'],
                low=hist['Low'],
                close=hist['Close'],
                name="OHLC"
            ),
            row=1, col=1
        )
    else:
        fig.add_trace(
            go.Scatter(
                x=hist.index,
                y=hist['Close'],
                mode='lines',
                name='Close',
                line=dict(color='blue', width=2)
            ),
            row=1, col=1
        )

    fig.add_trace(
        go.Scatter(x=hist.index, y=hist['MA20'],
                   line=dict(color='orange', width=1), name='MA20'),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=hist.index, y=hist['MA50'],
                   line=dict(color='green', width=1), name='MA50'),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=hist.index, y=hist['MA200'],
                   line=dict(color='red', width=1), name='MA200'),
        row=1, col=1
    )

    colors = ['red' if hist['Close'].iloc[i] < hist['Close'].iloc[i-1]
              else 'green' for i in range(1, len(hist))]
    colors.insert(0, 'grey')

    fig.add_trace(
        go.Bar(x=hist.index, y=hist['Volume'], name='Volume',
               marker_color=colors),
        row=2, col=1
    )

    delta = hist['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))

    fig.add_trace(
        go.Scatter(x=hist.index, y=rsi, name='RSI', line=dict(color='purple')),
        row=3, col=1
    )
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=3, col=1)
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=3, col=1)

    fig.update_layout(
        title=f'{ticker} - Исторические данные',
        yaxis_title='Цена ($)',
        xaxis_rangeslider_visible=False,
        height=800,
        showlegend=True,
        template='plotly_white'
    )

    fig.update_yaxes(title_text="Цена", row=1, col=1)
    fig.update_yaxes(title_text="Объем", row=2, col=1)
    fig.update_yaxes(title_text="RSI", row=3, col=1)

    return fig

def get_stock_info(ticker):
    """Получает основную информацию об акции"""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        basic_info = {
            'Название': info.get('longName', 'N/A'),
            'Сектор': info.get('sector', 'N/A'),
            'Индустрия': info.get('industry', 'N/A'),
            'Рыночная капитализация': f"${info.get('marketCap', 0):,}",
            'P/E': info.get('trailingPE', 'N/A'),
            'Дивиденд': f"{info.get('dividendYield', 0)*100:.2f}%" if info.get('dividendYield') else 'N/A',
            '52-нед макс': f"${info.get('fiftyTwoWeekHigh', 0):.2f}",
            '52-нед мин': f"${info.get('fiftyTwoWeekLow', 0):.2f}"
        }

        return basic_info
    except:
        return None

print("Функции для работы с акциями загружены")

Функции для работы с акциями загружены


In [14]:
# Словарь с предобученными моделями (вшитые ссылки)
PRETRAINED_MODELS = {
    'SBER': {
        'model_url': 'https://github.com/yourusername/stock-models/raw/main/sber_model.h5',
        'scaler_url': 'https://github.com/yourusername/stock-models/raw/main/sber_scaler.pkl',
        'description': 'Сбербанк, обучен на данных 2024-2025, lookback=60'
    },
    'GAZP': {
        'model_url': 'https://github.com/Kraftsan/predict_LSTM_MOEX/raw/master/model_GAZP_20260304_121835.h5',
        'scaler_url': 'https://github.com/Kraftsan/predict_LSTM_MOEX/raw/master/scaler_GAZP_20260304_121835.pkl',
        'description': 'Газпром, обучен на данных 2024-2025, lookback=60'
    },
    'LKOH': {
        'model_url': 'https://github.com/yourusername/stock-models/raw/main/lkoh_model.h5',
        'scaler_url': 'https://github.com/yourusername/stock-models/raw/main/lkoh_scaler.pkl',
        'description': 'Лукойл, обучен на данных 2024-2025, lookback=60'
    }
}

In [16]:
import requests
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import time
import joblib
from tensorflow.keras.models import load_model

try:
    from __main__ import get_moex_data_extended
    print("Импортирована функция get_moex_data_extended с пагинацией")
except:
    print("Не удалось импортировать get_moex_data_extended, использую локальную версию")

    def get_moex_data_extended(ticker, days=500):
        try:
            end_date = datetime.now()
            start_date = end_date - timedelta(days=days)
            start = start_date.strftime('%Y-%m-%d')
            end = end_date.strftime('%Y-%m-%d')

            print(f"Запрос к MOEX API для {ticker} за период {start} - {end}")

            all_rows = []
            start_value = 0
            page_size = 100

            while True:
                url = f"http://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/{ticker.upper()}.json"
                params = {
                    'from': start,
                    'till': end,
                    'start': start_value,
                    'limit': page_size,
                    'iss.meta': 'off',
                    'iss.only': 'history',
                    'history.columns': 'TRADEDATE,OPEN,HIGH,LOW,CLOSE,VOLUME'
                }

                headers = {'User-Agent': 'Mozilla/5.0'}
                response = requests.get(url, params=params, headers=headers, timeout=15)

                if response.status_code != 200:
                    break

                data = response.json()

                if 'history' not in data or not data['history'].get('data'):
                    break

                rows = data['history']['data']
                if not rows:
                    break

                all_rows.extend(rows)
                print(f"Получено {len(rows)} записей (всего: {len(all_rows)})")

                if len(rows) < page_size:
                    break

                start_value += page_size

            if not all_rows:
                return None, f"Нет данных по тикеру {ticker} на MOEX за период {start} - {end}"

            columns = data['history']['columns']
            df = pd.DataFrame(all_rows, columns=columns)

            rename_map = {
                'TRADEDATE': 'Date',
                'OPEN': 'Open',
                'HIGH': 'High',
                'LOW': 'Low',
                'CLOSE': 'Close',
                'VOLUME': 'Volume'
            }

            existing_cols = {k: v for k, v in rename_map.items() if k in df.columns}
            df = df.rename(columns=existing_cols)

            if 'Date' in df.columns:
                df['Date'] = pd.to_datetime(df['Date'])
                df = df.set_index('Date')

            for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            df = df.dropna(subset=['Open', 'High', 'Low', 'Close'])
            df = df.sort_index()

            print(f"Итого получено {len(df)} записей для {ticker}")
            print(f"Диапазон дат: {df.index[0]} - {df.index[-1]}")

            return df, None

        except Exception as e:
            return None, f"Ошибка: {str(e)}"

def validate_custom_model(model_file):
    """Проверяет загруженную пользователем модель"""
    try:
        # Сохраняем загруженный файл временно
        with open('temp_user_model.keras', 'wb') as f:
            f.write(model_file)

        model = load_model('temp_user_model.keras')

        # Проверки
        checks = []
        checks.append(("Файл загружен", True))

        # Проверка количества слоев
        if len(model.layers) >= 4:
            checks.append(("Количество слоев (≥4)", True))
        else:
            checks.append((f"Количество слоев ({len(model.layers)}), ожидалось ≥4", False))

        # Проверка входной формы
        expected_input = (None, 60, 1)
        if model.input_shape == expected_input:
            checks.append((f"Входная форма {model.input_shape}", True))
        else:
            checks.append((f"Входная форма {model.input_shape}, ожидалась {expected_input}", False))

        # Проверка выходной формы
        expected_output = (None, 1)
        if model.output_shape == expected_output:
            checks.append((f"Выходная форма {model.output_shape}", True))
        else:
            checks.append((f"Выходная форма {model.output_shape}, ожидалась {expected_output}", False))

        # Формируем результат
        result = "РЕЗУЛЬТАТ ВАЛИДАЦИИ МОДЕЛИ:\n" + "="*40 + "\n"
        all_passed = True
        for check, passed in checks:
            status = "✓" if passed else "✗"
            result += f"{status} {check}\n"
            if not passed:
                all_passed = False

        if all_passed:
            result += "\nМодель прошла валидацию! Можно использовать для предсказаний."
        else:
            result += "\nМодель не прошла валидацию. Проверьте требования."

        return result, model if all_passed else None

    except Exception as e:
        return f"ОШИБКА при загрузке модели: {str(e)}", None

def validate_custom_data(data_file):
    """Проверяет загруженный пользователем CSV файл"""
    try:
        # Сохраняем загруженный файл
        with open('temp_user_data.csv', 'wb') as f:
            f.write(data_file)

        df = pd.read_csv('temp_user_data.csv')

        checks = []
        checks.append(("Файл загружен", True))

        # Проверка обязательных колонок
        required_columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
        missing_cols = [col for col in required_columns if col not in df.columns]

        if not missing_cols:
            checks.append(("Все обязательные колонки присутствуют", True))
        else:
            checks.append((f"Отсутствуют колонки: {missing_cols}", False))

        # Проверка типов данных
        if 'Date' in df.columns:
            try:
                pd.to_datetime(df['Date'])
                checks.append(("Колонка Date - корректный формат даты", True))
            except:
                checks.append(("Колонка Date - неверный формат даты", False))

        for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
            if col in df.columns:
                if pd.api.types.is_numeric_dtype(df[col]):
                    checks.append((f"Колонка {col} - числовой формат", True))
                else:
                    checks.append((f"Колонка {col} - должен быть числовым", False))

        # Проверка количества данных
        if len(df) >= 200:
            checks.append((f"Количество записей: {len(df)} (≥200)", True))
        else:
            checks.append((f"Количество записей: {len(df)} (минимум 200)", False))

        # Проверка наличия пропусков
        if df[required_columns].isnull().sum().sum() == 0:
            checks.append(("Нет пропущенных значений", True))
        else:
            checks.append(("Есть пропущенные значения", False))

        # Формируем результат
        result = "РЕЗУЛЬТАТ ВАЛИДАЦИИ ДАННЫХ:\n" + "="*40 + "\n"
        all_passed = True
        for check, passed in checks:
            status = "✓" if passed else "✗"
            result += f"{status} {check}\n"
            if not passed:
                all_passed = False

        if all_passed:
            result += f"\nДанные прошли валидацию! Период: {df['Date'].min()} - {df['Date'].max()}"
            return result, df
        else:
            result += "\nДанные не прошли валидацию. Исправьте ошибки."
            return result, None

    except Exception as e:
        return f"ОШИБКА при загрузке данных: {str(e)}", None

def create_app():
    """Создает и возвращает приложение Gradio для анализа акций через MOEX с предсказаниями"""

    def get_moex_ticker_info(ticker):
        """Получает информацию о тикере на MOEX"""
        try:
            url = f"http://iss.moex.com/iss/securities/{ticker.upper()}.json"
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code == 200:
                data = response.json()

                if 'description' in data and data['description']['data']:
                    desc_data = data['description']['data']
                    desc_cols = data['description']['columns']

                    info = {}
                    for row in desc_data:
                        for i, col in enumerate(desc_cols):
                            if col == 'NAME' and i < len(row):
                                info['name'] = row[i]
                            if col == 'SHORTNAME' and i < len(row):
                                info['shortname'] = row[i]
                            if col == 'LATNAME' and i < len(row):
                                info['latname'] = row[i]

                    return info
            return {'name': ticker, 'shortname': ticker}
        except:
            return {'name': ticker, 'shortname': ticker}

    def create_chart(data, ticker):
        """Создает график акции MOEX"""
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            row_heights=[0.7, 0.3],
            subplot_titles=(
                f'{ticker} - цена (₽)',
                'Объем торгов'
            )
        )

        fig.add_trace(
            go.Candlestick(
                x=data.index,
                open=data['Open'],
                high=data['High'],
                low=data['Low'],
                close=data['Close'],
                name='OHLC',
                showlegend=False,
                increasing_line_color='green',
                decreasing_line_color='red'
            ),
            row=1, col=1
        )

        if len(data) >= 20:
            data['MA20'] = data['Close'].rolling(window=20).mean()
            fig.add_trace(
                go.Scatter(
                    x=data.index,
                    y=data['MA20'],
                    mode='lines',
                    name='MA20',
                    line=dict(color='orange', width=1.5, dash='dash')
                ),
                row=1, col=1
            )

        if len(data) >= 50:
            data['MA50'] = data['Close'].rolling(window=50).mean()
            fig.add_trace(
                go.Scatter(
                    x=data.index,
                    y=data['MA50'],
                    mode='lines',
                    name='MA50',
                    line=dict(color='blue', width=1.5, dash='dash')
                ),
                row=1, col=1
            )

        if 'Volume' in data.columns and len(data) > 1:
            colors = ['red' if data['Close'].iloc[i] < data['Close'].iloc[i-1]
                      else 'green' for i in range(1, len(data))]
            colors.insert(0, 'grey')

            fig.add_trace(
                go.Bar(
                    x=data.index,
                    y=data['Volume'],
                    name='Volume',
                    marker_color=colors,
                    showlegend=False
                ),
                row=2, col=1
            )

        fig.update_layout(
            height=700,
            template='plotly_white',
            xaxis_rangeslider_visible=False,
            title={
                'text': f'{ticker} - Московская биржа',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': 20}
            }
        )

        fig.update_yaxes(title_text="Цена (₽)", row=1, col=1)
        fig.update_yaxes(title_text="Объем", row=2, col=1)
        fig.update_xaxes(title_text="Дата", row=2, col=1)

        return fig

    def analyze_stock(ticker, period):
        """Основная функция анализа MOEX акций"""
        if not ticker:
            return None, "Введите тикер акции MOEX", None

        ticker = ticker.upper().strip()

        period_map = {
            '1mo': 30,
            '3mo': 90,
            '6mo': 180,
            '1y': 365,
            '2y': 730
        }
        days = period_map.get(period, 500)

        data, error = get_moex_data_extended(ticker, days)

        if error:
            return None, f"{error}", None

        if data is None or len(data) == 0:
            return None, "Нет данных за выбранный период", None

        fig = create_chart(data.copy(), ticker)
        ticker_info = get_moex_ticker_info(ticker)

        last = data.iloc[-1]
        prev = data.iloc[-2] if len(data) > 1 else last
        first = data.iloc[0]

        change_day = last['Close'] - prev['Close']
        change_day_pct = (change_day / prev['Close']) * 100 if prev['Close'] != 0 else 0
        change_total = last['Close'] - first['Close']
        change_total_pct = (change_total / first['Close']) * 100 if first['Close'] != 0 else 0
        volume_avg = data['Volume'].mean() if 'Volume' in data.columns else 0
        returns = data['Close'].pct_change().dropna()
        volatility = returns.std() * (252 ** 0.5) * 100 if len(returns) > 0 else 0

        stats = f"""СТАТИСТИКА {ticker} (MOEX)

Текущая цена: {last['Close']:.2f} ₽
Изменение за день: {change_day:+.2f} ₽ ({change_day_pct:+.2f}%)
Изменение за период: {change_total:+.2f} ₽ ({change_total_pct:+.2f}%)

Максимум: {data['High'].max():.2f} ₽
Минимум: {data['Low'].min():.2f} ₽
Средний объем: {volume_avg:,.0f} шт.

Волатильность (год): {volatility:.1f}%
Торговых дней: {len(data)}
        """

        company_name = ticker_info.get('shortname', ticker) if ticker_info else ticker

        info = f"""ИНФОРМАЦИЯ О ТИКЕРЕ

Тикер: {ticker}
Компания: {company_name}
Биржа: Московская биржа (MOEX)
Период: {period}
Начало: {data.index[0].strftime('%Y-%m-%d')}
Конец: {data.index[-1].strftime('%Y-%m-%d')}
        """

        tech_info = []
        if 'MA20' in data.columns and not data['MA20'].isna().all():
            tech_info.append(f"MA20 (20 дней): {data['MA20'].iloc[-1]:.2f} ₽")
        if 'MA50' in data.columns and not data['MA50'].isna().all():
            tech_info.append(f"MA50 (50 дней): {data['MA50'].iloc[-1]:.2f} ₽")

        if tech_info:
            info += "\n\nТехнические индикаторы:\n" + "\n".join(tech_info)

        return fig, stats, info

    with gr.Blocks(title="Анализ акций MOEX", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# Анализ акций Московской биржи (MOEX)")
        gr.Markdown("Введите тикер для получения данных с MOEX")

        with gr.Row():
            with gr.Column(scale=1):
                ticker = gr.Textbox(
                    label="Тикер акции",
                    placeholder="SBER, GAZP, LKOH, YNDX...",
                    value="SBER"
                )

                period = gr.Dropdown(
                    choices=["1mo", "3mo", "6mo", "1y", "2y"],
                    value="1y",
                    label="Период для анализа"
                )

                with gr.Row():
                    analyze_btn = gr.Button("Анализировать", variant="secondary")
                    predict_btn = gr.Button("Предсказать (ML)", variant="primary")

                gr.Markdown("---")
                gr.Markdown("### Настройки модели")

                # Чекбокс для использования предобученной модели
                use_pretrained = gr.Checkbox(
                    label="Использовать предобученную модель из облака",
                    value=False,
                    info="Загружает готовую модель по вшитой ссылке"
                )

                # Информация о доступных моделях
                model_info = gr.Markdown(
                    "**Доступные предобученные модели:**\n" +
                    "\n".join([f"- {ticker}: {info['description']}" for ticker, info in PRETRAINED_MODELS.items()])
                )

                gr.Markdown("---")
                gr.Markdown("### Загрузка своих данных")

                # Загрузка своей модели
                upload_model = gr.File(
                    label="Загрузить свою модель (.keras)",
                    file_types=[".keras"]
                )

                validate_model_btn = gr.Button("Проверить модель", variant="secondary")
                model_validation_output = gr.Textbox(label="Результат проверки модели", lines=8)

                # Загрузка своих данных
                upload_data = gr.File(
                    label="Загрузить свои данные (CSV)",
                    file_types=[".csv"]
                )

                validate_data_btn = gr.Button("Проверить данные", variant="secondary")
                data_validation_output = gr.Textbox(label="Результат проверки данных", lines=10)

                # Ссылки на скачивание (появятся после обучения)
                download_files = gr.Column(visible=False)
                with download_files:
                    gr.Markdown("### Скачать обученную модель")
                    download_model = gr.File(label="Скачать модель", visible=False)
                    download_scaler = gr.File(label="Скачать scaler", visible=False)

            with gr.Column(scale=2):
                plot = gr.Plot(label="График")

        with gr.Row():
            with gr.Column():
                stats = gr.Markdown(label="Статистика")
            with gr.Column():
                info = gr.Markdown(label="Информация")

        with gr.Row():
            with gr.Column():
                metrics_output = gr.Markdown(label="Метрики модели")
            with gr.Column():
                training_output = gr.Markdown(label="Процесс обучения")

        with gr.Row():
            prediction_output = gr.Markdown(label="Предсказание")

        gr.Markdown("### Популярные тикеры MOEX")
        examples = [
            ["SBER", "1y"],
            ["GAZP", "1y"],
            ["LKOH", "1y"],
            ["YNDX", "1y"],
            ["MGNT", "1y"],
            ["ROSN", "1y"]
        ]

        gr.Examples(
            examples=examples,
            inputs=[ticker, period],
            outputs=[plot, stats, info],
            fn=analyze_stock,
            cache_examples=False
        )

        # Обработчики для валидации
        validate_model_btn.click(
            fn=validate_custom_model,
            inputs=[upload_model],
            outputs=[model_validation_output]
        )

        validate_data_btn.click(
            fn=validate_custom_data,
            inputs=[upload_data],
            outputs=[data_validation_output]
        )

        # Обработчики для анализа
        analyze_btn.click(
            fn=analyze_stock,
            inputs=[ticker, period],
            outputs=[plot, stats, info]
        )

        ticker.submit(
            fn=analyze_stock,
            inputs=[ticker, period],
            outputs=[plot, stats, info]
        )

        # Обработчик для предсказания с учетом чекбокса
        def predict_with_model(ticker, period, use_pretrained):
            if use_pretrained and ticker in PRETRAINED_MODELS:
                # Используем предобученную модель из облака
                return predict_with_pretrained(ticker, period)
            else:
                # Обучаем новую модель
                return train_and_predict(ticker, period)

        try:
            from __main__ import train_and_predict, predict_with_pretrained
            predict_fn = predict_with_model
            print("Модель LSTM загружена")
        except:
            def predict_fn(ticker, period, use_pretrained):
                return None, "Модель не загружена", None, None, None, None
            print("Модель не найдена")

        predict_btn.click(
            fn=predict_fn,
            inputs=[ticker, period, use_pretrained],
            outputs=[plot, metrics_output, training_output, prediction_output, download_model, download_scaler]
        )

    return demo

demo = create_app()
print("Приложение с MOEX API")

try:
    stop_server()
except:
    pass
start_server()

Импортирована функция get_moex_data_extended с пагинацией
Модель LSTM загружена
Приложение с MOEX API
Closing server running on port: 7860
Сервер остановлен
Сервер остановлен

Сервер запущен на порту 7860
http://127.0.0.1:7860
Публичная ссылка появится выше


7860

In [3]:
import threading
import time
from IPython.display import display, HTML

server_thread = None
current_demo = None
current_port = None

def start_server(app=None, port=7860, share=True):
    """Запускает сервер Gradio"""
    global server_thread, current_demo, current_port

    stop_server()

    for p in [port, port+1, port+2]:
        kill_port(p)

    if app is None:
        try:
            app = demo
        except:
            print("Приложение не найдено")
            return

    current_demo = app
    current_port = get_free_port(port)

    def _run():
        app.launch(
            share=share,
            server_name='0.0.0.0',
            server_port=current_port,
            quiet=True,
            prevent_thread_lock=True
        )

    server_thread = threading.Thread(target=_run)
    server_thread.daemon = True
    server_thread.start()

    time.sleep(2)

    print(f"\nСервер запущен на порту {current_port}")
    print(f"http://127.0.0.1:{current_port}")
    if share:
        print("Публичная ссылка появится выше")

    display(HTML(f'''
    <div style="background:#4CAF50; color:white; padding:10px; border-radius:5px; margin:5px 0;">
        <b>Сервер работает</b><br>
        <a href="http://127.0.0.1:{current_port}" target="_blank" style="color:yellow;">
            http://127.0.0.1:{current_port}
        </a>
    </div>
    '''))

    return current_port

def stop_server():
    """Останавливает сервер"""
    global current_demo
    if current_demo:
        try:
            current_demo.close()
            print("Сервер остановлен")
        except:
            pass

def restart_server():
    """Перезапускает сервер с текущим приложением"""
    global current_demo, current_port
    if current_demo:
        port = current_port or 7860
        start_server(current_demo, port)
    else:
        print("Нет запущенного сервера")

print("Менеджер сервера загружен")
print("\nКоманды:")
print("  start_server()    - запустить сервер")
print("  stop_server()     - остановить сервер")
print("  restart_server()  - перезапустить сервер")

Менеджер сервера загружен

Команды:
  start_server()    - запустить сервер
  stop_server()     - остановить сервер
  restart_server()  - перезапустить сервер


In [15]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')
import requests
from datetime import datetime, timedelta
import joblib
import os
import urllib.request

class LossHistory(Callback):
    def __init__(self):
        super().__init__()
        self.epochs = []
        self.losses = []
        self.val_losses = []

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}
        self.epochs.append(epoch + 1)
        self.losses.append(logs.get('loss'))
        self.val_losses.append(logs.get('val_loss'))
        if (epoch + 1) % 10 == 0:
            print(f"Эпоха {epoch + 1}: loss = {logs.get('loss'):.6f}, val_loss = {logs.get('val_loss'):.6f}")

def download_pretrained_model(ticker):
    """Скачивает предобученную модель и scaler из облака"""
    try:
        if ticker not in PRETRAINED_MODELS:
            return None, None, f"Нет предобученной модели для тикера {ticker}"

        model_info = PRETRAINED_MODELS[ticker]

        # Скачиваем модель
        print(f"Скачивание модели для {ticker}...")
        model_path = f'pretrained_{ticker}_model.keras'
        urllib.request.urlretrieve(model_info['model_url'], model_path)

        # Скачиваем scaler
        print(f"Скачивание scaler для {ticker}...")
        scaler_path = f'pretrained_{ticker}_scaler.pkl'
        urllib.request.urlretrieve(model_info['scaler_url'], scaler_path)

        # Загружаем модель и scaler
        model = load_model(model_path)
        scaler = joblib.load(scaler_path)

        print(f"✅ Предобученная модель для {ticker} загружена")
        return model, scaler, None

    except Exception as e:
        return None, None, f"Ошибка загрузки предобученной модели: {str(e)}"

def get_moex_data_extended(ticker, days=500):
    """Получает данные через MOEX API с пагинацией"""
    try:
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days)
        start = start_date.strftime('%Y-%m-%d')
        end = end_date.strftime('%Y-%m-%d')

        print(f"Запрос к MOEX API для {ticker} за период {start} - {end}")

        all_rows = []
        start_value = 0
        page_size = 100

        while True:
            url = f"http://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/{ticker.upper()}.json"
            params = {
                'from': start,
                'till': end,
                'start': start_value,
                'limit': page_size,
                'iss.meta': 'off',
                'iss.only': 'history',
                'history.columns': 'TRADEDATE,OPEN,HIGH,LOW,CLOSE,VOLUME'
            }

            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, params=params, headers=headers, timeout=15)

            if response.status_code != 200:
                break

            data = response.json()

            if 'history' not in data or not data['history'].get('data'):
                break

            rows = data['history']['data']
            if not rows:
                break

            all_rows.extend(rows)
            print(f"Получено {len(rows)} записей (всего: {len(all_rows)})")

            if len(rows) < page_size:
                break

            start_value += page_size

        if not all_rows:
            return None, f"Нет данных по тикеру {ticker} на MOEX за период {start} - {end}"

        columns = data['history']['columns']
        df = pd.DataFrame(all_rows, columns=columns)

        rename_map = {
            'TRADEDATE': 'Date',
            'OPEN': 'Open',
            'HIGH': 'High',
            'LOW': 'Low',
            'CLOSE': 'Close',
            'VOLUME': 'Volume'
        }

        existing_cols = {k: v for k, v in rename_map.items() if k in df.columns}
        df = df.rename(columns=existing_cols)

        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.set_index('Date')

        for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        df = df.dropna(subset=['Open', 'High', 'Low', 'Close'])
        df = df.sort_index()

        print(f"Итого получено {len(df)} записей для {ticker}")
        print(f"Диапазон дат: {df.index[0]} - {df.index[-1]}")

        return df, None

    except Exception as e:
        return None, f"Ошибка: {str(e)}"

def prepare_lstm_data(data, lookback=60):
    """Подготавливает данные для LSTM"""
    prices = data['Close'].values.reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_prices = scaler.fit_transform(prices)

    X, y = [], []
    for i in range(lookback, len(scaled_prices)):
        X.append(scaled_prices[i-lookback:i, 0])
        y.append(scaled_prices[i, 0])

    X = np.array(X).reshape(-1, lookback, 1)
    y = np.array(y)
    return X, y, scaler

def create_lstm_model(lookback=60):
    """Создает LSTM модель"""
    model = Sequential([
        LSTM(150, return_sequences=True, input_shape=(lookback, 1)),
        Dropout(0.3),
        LSTM(150, return_sequences=True),
        Dropout(0.3),
        LSTM(100, return_sequences=True),
        Dropout(0.3),
        LSTM(50, return_sequences=False),
        Dropout(0.2),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='mse',
        metrics=['mae']
    )
    return model

def train_model(data, lookback=60, epochs=100, batch_size=32):
    """Обучение модели с сохранением"""
    X, y, scaler = prepare_lstm_data(data, lookback)

    if len(X) < 100:
        print(f"Мало данных для обучения: {len(X)} последовательностей")
        return None, None, None, None, None

    train_size = int(len(X) * 0.8)
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]

    print(f"Размер обучающей выборки: {len(X_train)}")
    print(f"Размер тестовой выборки: {len(X_test)}")

    model = create_lstm_model(lookback)
    history_callback = LossHistory()

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        mode='min'
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=0.00001,
        mode='min',
        verbose=0
    )

    print("Начало обучения модели...")

    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[history_callback, early_stopping, reduce_lr]
    )

    print(f"Обучение завершено. Всего эпох: {len(history_callback.epochs)}")

    train_pred = model.predict(X_train, verbose=0)
    test_pred = model.predict(X_test, verbose=0)

    train_pred_rescaled = scaler.inverse_transform(train_pred)
    test_pred_rescaled = scaler.inverse_transform(test_pred)

    train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))
    test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

    train_rmse = np.sqrt(mean_squared_error(train_actual, train_pred_rescaled))
    test_rmse = np.sqrt(mean_squared_error(test_actual, test_pred_rescaled))
    train_mae = mean_absolute_error(train_actual, train_pred_rescaled)
    test_mae = mean_absolute_error(test_actual, test_pred_rescaled)
    train_mape = np.mean(np.abs((train_actual - train_pred_rescaled) / train_actual)) * 100
    test_mape = np.mean(np.abs((test_actual - test_pred_rescaled) / test_actual)) * 100
    train_max_error = np.max(np.abs(train_actual - train_pred_rescaled))
    test_max_error = np.max(np.abs(test_actual - test_pred_rescaled))
    train_r2 = r2_score(train_actual, train_pred_rescaled)
    test_r2 = r2_score(test_actual, test_pred_rescaled)

    metrics = {
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_mape': train_mape,
        'test_mape': test_mape,
        'train_max_error': train_max_error,
        'test_max_error': test_max_error,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_accuracy': max(0, 100 - train_mape),
        'test_accuracy': max(0, 100 - test_mape)
    }

    print(f"\nСтатистика ошибок:")
    print(f"Обучающая выборка - RMSE: {train_rmse:.2f}₽, MAE: {train_mae:.2f}₽, Max Error: {train_max_error:.2f}₽")
    print(f"Тестовая выборка   - RMSE: {test_rmse:.2f}₽, MAE: {test_mae:.2f}₽, Max Error: {test_max_error:.2f}₽")

    return model, scaler, history_callback, metrics, (X, y)

def predict_future(model, scaler, data, lookback=60, days=90, n_iterations=10):
    """Предсказание с ансамблевым подходом"""
    last_prices = data['Close'].values
    last_sequence = last_prices[-lookback:].reshape(-1, 1)
    last_sequence_scaled = scaler.transform(last_sequence)

    all_predictions = []

    for _ in range(n_iterations):
        predictions = []
        current_sequence = last_sequence_scaled.copy()

        for _ in range(days):
            X_pred = current_sequence.reshape(1, lookback, 1)
            noise = np.random.normal(0, 0.01, X_pred.shape)
            X_pred_noisy = X_pred + noise
            next_pred_scaled = model.predict(X_pred_noisy, verbose=0)[0, 0]
            predictions.append(next_pred_scaled)
            current_sequence = np.roll(current_sequence, -1)
            current_sequence[-1] = next_pred_scaled

        predictions_rescaled = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
        all_predictions.append(predictions_rescaled)

    ensemble_predictions = np.mean(all_predictions, axis=0)
    ensemble_std = np.std(all_predictions, axis=0)

    return ensemble_predictions, ensemble_std

def create_prediction_chart(data, ticker, model, scaler, metrics, lookback=60, prediction_days=90):
    """Создает график с предсказаниями"""
    future_predictions, future_std = predict_future(model, scaler, data, lookback, prediction_days)
    last_date = data.index[-1]
    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=prediction_days, freq='D')

    validation_days = min(90, len(data) // 4)
    val_predictions = []
    val_actual = []
    val_dates = []
    val_errors = []

    if len(data) > validation_days + lookback:
        val_actual = data['Close'].values[-validation_days:]
        val_dates = data.index[-validation_days:]

        for i in range(validation_days):
            hist_end = len(data) - validation_days + i
            if hist_end >= lookback:
                hist_data = data.iloc[:hist_end]
                val_pred, _ = predict_future(model, scaler, hist_data, lookback, 1)
                val_predictions.append(val_pred[0])
                val_errors.append(abs(val_actual[i] - val_pred[0]))
            else:
                val_predictions.append(np.nan)
                val_errors.append(np.nan)

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.5, 0.25, 0.25],
        subplot_titles=(
            f'{ticker} - Цена с предсказанием',
            'Ошибка предсказания',
            'Изменение по неделям'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=data.index,
            y=data['Close'],
            mode='lines',
            name='Исторические данные',
            line=dict(color='blue', width=2)
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=future_dates,
            y=future_predictions,
            mode='lines',
            name='Предсказание',
            line=dict(color='red', width=2, dash='dash')
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=future_dates,
            y=future_predictions + 2*future_std,
            mode='lines',
            name='+2σ',
            line=dict(color='red', width=1, dash='dot')
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=future_dates,
            y=future_predictions - 2*future_std,
            mode='lines',
            name='-2σ',
            line=dict(color='red', width=1, dash='dot'),
            fill='tonexty',
            fillcolor='rgba(255,0,0,0.1)'
        ),
        row=1, col=1
    )

    if len(val_predictions) > 0 and not all(np.isnan(val_predictions)):
        fig.add_trace(
            go.Scatter(
                x=val_dates,
                y=val_actual,
                mode='lines',
                name='Реальные (3 мес)',
                line=dict(color='green', width=2)
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=val_dates,
                y=val_predictions,
                mode='lines',
                name='Ретро-прогноз',
                line=dict(color='orange', width=2, dash='dot')
            ),
            row=1, col=1
        )

        valid_idx = ~np.isnan(val_errors)
        if np.sum(valid_idx) > 0:
            fig.add_trace(
                go.Bar(
                    x=val_dates[valid_idx],
                    y=np.array(val_errors)[valid_idx],
                    name='Ошибка',
                    marker_color='purple',
                    opacity=0.6
                ),
                row=2, col=1
            )

    weekly_changes = []
    weekly_labels = []
    for i in range(0, prediction_days, 7):
        if i + 7 <= prediction_days:
            change = ((future_predictions[i+6] - future_predictions[i]) / future_predictions[i]) * 100
            weekly_changes.append(change)
            weekly_labels.append(f'{i//7 + 1}')

    colors = ['green' if c > 0 else 'red' for c in weekly_changes]
    fig.add_trace(
        go.Bar(
            x=weekly_labels,
            y=weekly_changes,
            name='Изменение %',
            marker_color=colors,
            text=[f'{c:.1f}%' for c in weekly_changes],
            textposition='outside'
        ),
        row=3, col=1
    )

    fig.add_hline(y=0, line_dash="solid", line_color="black", row=3, col=1)

    fig.update_layout(
        height=900,
        title_text=f"Анализ и предсказание {ticker}",
        showlegend=True,
        template='plotly_white',
        hovermode='x unified'
    )

    fig.update_xaxes(title_text="Дата", row=1, col=1)
    fig.update_xaxes(title_text="Дата", row=2, col=1)
    fig.update_xaxes(title_text="Неделя", row=3, col=1)

    fig.update_yaxes(title_text="Цена (₽)", row=1, col=1)
    fig.update_yaxes(title_text="Ошибка (₽)", row=2, col=1)
    fig.update_yaxes(title_text="Изменение (%)", row=3, col=1)

    fig.add_vline(x=last_date, line_width=2, line_dash="dash", line_color="black", row=1, col=1)

    return fig

def save_model_files(model, scaler, ticker):
    """Сохраняет модель и scaler в файлы"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_filename = f'model_{ticker}_{timestamp}.keras'
    scaler_filename = f'scaler_{ticker}_{timestamp}.pkl'

    # Сохраняем
    model.save(model_filename)
    joblib.dump(scaler, scaler_filename)

    print(f"Модель сохранена как: {model_filename}")
    print(f"Scaler сохранен как: {scaler_filename}")

    return model_filename, scaler_filename

def train_and_predict(ticker, period):
    """Основная функция обучения и предсказания"""
    if not ticker:
        return None, "Введите тикер акции", None, None, None, None

    ticker = ticker.upper().strip()

    print(f"Загрузка данных для {ticker} за 2 года...")
    data, error = get_moex_data_extended(ticker, days=500)

    if error:
        return None, f"Ошибка загрузки данных: {error}", None, None, None, None

    if data is None or len(data) < 200:
        return None, f"Недостаточно данных. Получено {len(data) if data is not None else 0} дней", None, None, None, None

    print(f"Загружено {len(data)} дней данных для обучения")
    print(f"Период: {data.index[0].strftime('%Y-%m-%d')} - {data.index[-1].strftime('%Y-%m-%d')}")

    lookback = min(60, len(data) // 5)
    epochs = 100
    batch_size = 32

    print(f"Обучение модели с lookback={lookback}...")

    try:
        model, scaler, history, metrics, _ = train_model(
            data, lookback=lookback, epochs=epochs, batch_size=batch_size
        )

        if model is None:
            return None, "Не удалось обучить модель", None, None, None, None

        # Сохраняем модель и scaler
        model_file, scaler_file = save_model_files(model, scaler, ticker)

        fig = create_prediction_chart(data, ticker, model, scaler, metrics, lookback, prediction_days=90)

        metrics_report = f"""МЕТРИКИ МОДЕЛИ LSTM (обучение на {len(data)} днях)

Обучающая выборка (80% данных):
- RMSE: {metrics['train_rmse']:.2f} ₽
- MAE: {metrics['train_mae']:.2f} ₽
- Max Error: {metrics['train_max_error']:.2f} ₽
- MAPE: {metrics['train_mape']:.2f}%
- R²: {metrics['train_r2']:.3f}
- Точность: {metrics['train_accuracy']:.2f}%

Тестовая выборка (20% данных):
- RMSE: {metrics['test_rmse']:.2f} ₽
- MAE: {metrics['test_mae']:.2f} ₽
- Max Error: {metrics['test_max_error']:.2f} ₽
- MAPE: {metrics['test_mape']:.2f}%
- R²: {metrics['test_r2']:.3f}
- Точность: {metrics['test_accuracy']:.2f}%

Параметры модели:
- Архитектура: LSTM (4 слоя) с Dropout 0.3
- Lookback: {lookback} дней
- Эпохи: {len(history.epochs)}
- Всего данных: {len(data)} торговых дней
- Период: {data.index[0].strftime('%Y-%m-%d')} - {data.index[-1].strftime('%Y-%m-%d')}

Анализ ошибок:
- Тестовая MAE: {metrics['test_mae']:.2f} ₽ ({metrics['test_mae']/data['Close'].mean()*100:.2f}% от средней цены)
- Максимальная ошибка: {metrics['test_max_error']:.2f} ₽
- 95% предсказаний попадают в интервал ±{metrics['test_rmse']*2:.2f} ₽
        """

        training_history = "ПРОЦЕСС ОБУЧЕНИЯ\n\n"
        training_history += "Эпоха | Loss | Val Loss\n"
        training_history += "-----|------|----------\n"

        for i in range(min(20, len(history.epochs))):
            training_history += f"{history.epochs[i]} | {history.losses[i]:.6f} | {history.val_losses[i]:.6f}\n"

        if len(history.epochs) > 20:
            training_history += "... | ... | ...\n"
            training_history += f"{history.epochs[-1]} | {history.losses[-1]:.6f} | {history.val_losses[-1]:.6f}\n"

        last_price = data['Close'].iloc[-1]
        future_pred, future_std = predict_future(model, scaler, data, lookback, 90)

        changes = []
        for i in [29, 59, 89]:
            change_pct = (future_pred[i] - last_price) / last_price * 100
            changes.append(change_pct)

        prediction_report = f"""ПРЕДСКАЗАНИЕ НА 90 ДНЕЙ

Текущая цена: {last_price:.2f} ₽

Прогноз с доверительным интервалом (95%):
- Через 1 месяц: {future_pred[29]:.2f} ₽ [{future_pred[29] - 2*future_std[29]:.2f} - {future_pred[29] + 2*future_std[29]:.2f}] ₽
- Через 2 месяца: {future_pred[59]:.2f} ₽ [{future_pred[59] - 2*future_std[59]:.2f} - {future_pred[59] + 2*future_std[59]:.2f}] ₽
- Через 3 месяца: {future_pred[89]:.2f} ₽ [{future_pred[89] - 2*future_std[89]:.2f} - {future_pred[89] + 2*future_std[89]:.2f}] ₽

Изменение:
- Через 1 месяц: {changes[0]:+.2f}% ±{2*future_std[29]/last_price*100:.2f}%
- Через 2 месяца: {changes[1]:+.2f}% ±{2*future_std[59]/last_price*100:.2f}%
- Через 3 месяца: {changes[2]:+.2f}% ±{2*future_std[89]/last_price*100:.2f}%

Статистика предсказания:
- Минимум: {min(future_pred):.2f} ₽
- Максимум: {max(future_pred):.2f} ₽
- Средняя волатильность: {np.mean(future_std/future_pred)*100:.2f}%
- Доверительный интервал: ±{np.mean(2*future_std):.2f} ₽
        """

        return fig, metrics_report, training_history, prediction_report, model_file, scaler_file

    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"Ошибка при обучении модели: {str(e)}", None, None, None, None


def predict_with_pretrained(ticker, period):
    """Использует предобученную модель из облака"""
    if not ticker:
        return None, "Введите тикер акции", None, None, None, None

    ticker = ticker.upper().strip()

    # Скачиваем предобученную модель
    model, scaler, error = download_pretrained_model(ticker)
    if error:
        return None, error, None, None, None, None

    # Загружаем свежие данные для предсказания
    print(f"Загрузка свежих данных для {ticker}...")
    data, error = get_moex_data_extended(ticker, days=500)

    if error:
        return None, f"Ошибка загрузки данных: {error}", None, None, None, None

    # Создаем график с предсказаниями
    lookback = 60  # Должен совпадать с lookback при обучении
    fig = create_prediction_chart(data, ticker, model, scaler, {
        'test_rmse': 0, 'test_mae': 0, 'test_mape': 0, 'test_r2': 0, 'test_accuracy': 0
    }, lookback, prediction_days=90)

    # Делаем предсказание
    last_price = data['Close'].iloc[-1]
    future_pred, future_std = predict_future(model, scaler, data, lookback, 90)

    changes = []
    for i in [29, 59, 89]:
        change_pct = (future_pred[i] - last_price) / last_price * 100
        changes.append(change_pct)

    prediction_report = f"""ПРЕДСКАЗАНИЕ НА 90 ДНЕЙ (предобученная модель)

Текущая цена: {last_price:.2f} ₽

Прогноз с доверительным интервалом (95%):
- Через 1 месяц: {future_pred[29]:.2f} ₽ [{future_pred[29] - 2*future_std[29]:.2f} - {future_pred[29] + 2*future_std[29]:.2f}] ₽
- Через 2 месяца: {future_pred[59]:.2f} ₽ [{future_pred[59] - 2*future_std[59]:.2f} - {future_pred[59] + 2*future_std[59]:.2f}] ₽
- Через 3 месяца: {future_pred[89]:.2f} ₽ [{future_pred[89] - 2*future_std[89]:.2f} - {future_pred[89] + 2*future_std[89]:.2f}] ₽

Изменение:
- Через 1 месяц: {changes[0]:+.2f}% ±{2*future_std[29]/last_price*100:.2f}%
- Через 2 месяца: {changes[1]:+.2f}% ±{2*future_std[59]/last_price*100:.2f}%
- Через 3 месяца: {changes[2]:+.2f}% ±{2*future_std[89]/last_price*100:.2f}%
    """

    metrics_report = f"""ИСПОЛЬЗОВАНА ПРЕДОБУЧЕННАЯ МОДЕЛЬ

Тикер: {ticker}
Модель: {PRETRAINED_MODELS[ticker]['description']}
Дата предсказания: {datetime.now().strftime('%Y-%m-%d %H:%M')}

Для получения метрик обучите модель заново (снимите чекбокс).
    """

    training_history = "ПРЕДОБУЧЕННАЯ МОДЕЛЬ\n\nПроцесс обучения не отображается для предобученной модели."

    return fig, metrics_report, training_history, prediction_report, None, None

print("Модель LSTM загружена с поддержкой сохранения и загрузки")
print("Используйте train_and_predict(ticker, period) для обучения")
print("Используйте predict_with_pretrained(ticker, period) для предобученной модели")

Модель LSTM загружена с поддержкой сохранения и загрузки
Используйте train_and_predict(ticker, period) для обучения
Используйте predict_with_pretrained(ticker, period) для предобученной модели
